# Phase 7 — Interactive Decision-Support Dashboard (Streamlit & Full Exports)

**Project:** EcoFarm Circular — AI-Based Intelligent Energy Management System  
**Provisional Site:** Settat, Casablanca-Settat, Morocco (`33.0010° N, -7.6166° E`)

## Learning Objectives
1. Audit all Phase 2–6 datasets, `.pkl` models, and validation reports using `dashboard.utils.validation`.
2. Verify missing-file, missing-model, out-of-range, `NaN`/`Inf`, bridge parity, and PLC safety lock checks (`verify_phase7_dashboard.py`).
3. Launch the 9-section Streamlit decision-support dashboard (`EcoFarm_AI/dashboard/app.py`) in Google Colab (via Cloudflare tunnel) or locally.
4. Export and download the complete Phase 7 ZIP archive + master `DATA_DICTIONARY.csv` and `PROJECT_MANIFEST.csv`.


In [ ]:
# Google Colab & Portable Environment Bootstrap (Relative Paths + Optional Drive/Upload)
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit>=1.32.0', 'plotly>=5.18.0'])

# Option A (Recommended in Colab): Mount Google Drive if EcoFarm_AI is saved on Drive
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust folder name below if stored elsewhere in Google Drive:
    os.chdir('/content/drive/MyDrive/EcoFarm_AI')

# Resolve EcoFarm_AI project root reliably from notebooks/, root, or Colab /content/
cwd = Path.cwd().resolve()
if (cwd / 'src').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / 'EcoFarm_AI' / 'src').is_dir():
    PROJECT_ROOT = cwd / 'EcoFarm_AI'
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Environment   :', 'Google Colab' if IN_COLAB else 'Local / Jupyter')
print('PROJECT_ROOT  :', PROJECT_ROOT)


In [ ]:
# Dataset & Module Availability Check (with Optional Colab Upload Prompt)
required_files = [
    'dashboard/app.py',
    'src/verify_phase7_dashboard.py',
    'DATA_DICTIONARY.csv',
    'PROJECT_MANIFEST.csv',
]
missing = [p for p in required_files if not (PROJECT_ROOT / p).is_file()]
print(f'Verified {len(required_files) - len(missing)}/{len(required_files)} required project files in {PROJECT_ROOT}.')

if missing:
    print('Missing relative files:', missing)
    if IN_COLAB:
        from google.colab import files
        print('Upload the missing EcoFarm CSV / Phase ZIP files below:')
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall(PROJECT_ROOT.parent if PROJECT_ROOT.name == 'EcoFarm_AI' else PROJECT_ROOT)
                print('Extracted uploaded archive:', fname)


In [ ]:
from dashboard.utils.validation import audit_phase_artifacts
from src.verify_phase7_dashboard import run_phase7_verification

artifacts = audit_phase_artifacts(PROJECT_ROOT)
for k, item in artifacts.items():
    print(f'[{item.status_label:19s}] {item.label:<44s} ({item.size_bytes:,} bytes)')

rep = run_phase7_verification(write_workbench_outputs=False)
print(f"\nAutomated Verification Score: {rep['tests_passed']} / {rep['tests_total']} PASSED")
for t in rep['tests']:
    print(f"  [{t['status']}] {t['test_id']} {t['name']}")


In [ ]:
# Launch Streamlit Decision-Support Dashboard
import time
if IN_COLAB:
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_ROOT)
    subprocess.Popen(
        ['streamlit', 'run', str(PROJECT_ROOT / 'dashboard' / 'app.py'), '--server.port', '8501', '--server.headless', 'true'],
        env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(3)
    print('Starting Cloudflare tunnel for Streamlit (port 8501)... Stop cell when finished.')
    !cloudflared tunnel --url http://localhost:8501
else:
    print(f'Run locally: PYTHONPATH={PROJECT_ROOT} streamlit run {PROJECT_ROOT / "dashboard" / "app.py"} --server.port 8501')


## Conclusions & Limitations
- Dashboard UI bridges directly to `src/` physics & ML modules (`0.0e+00 kW_th` deviation vs. `src.thermal_model`).
- Dashboard controls remain advisory simulation tools only and cannot bypass hardwired PLC interlocks (`600 °C` trip).


In [ ]:
# Export & Download Phase 7 Package (CSV, XLSX, Notebook, Code, Figures & Reports)
export_dir = PROJECT_ROOT / 'exports' / 'phase_07'
export_dir.mkdir(parents=True, exist_ok=True)
zip_path = export_dir / 'EcoFarm_Phase07_Package.zip'

patterns = ['dashboard/**/*.py', 'src/verify_phase7_dashboard.py', 'DATA_DICTIONARY.csv', 'PROJECT_MANIFEST.csv', 'data/outputs/phase7_*.csv', 'reports/phase7_*', 'notebooks/06_dashboard.ipynb']
added = set()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for fp in sorted(PROJECT_ROOT.glob(pat)):
            if fp.is_file() and '__pycache__' not in fp.parts and fp not in added:
                added.add(fp)
                arc = (Path('EcoFarm_AI') / fp.relative_to(PROJECT_ROOT)).as_posix()
                zi = zipfile.ZipInfo(arc, date_time=(2026, 11, 21, 12, 0, 0))
                zi.create_system = 0
                zi.compress_type = zipfile.ZIP_DEFLATED
                zi.external_attr = 0o644 << 16
                zf.writestr(zi, fp.read_bytes())

with zipfile.ZipFile(zip_path, 'r') as zcheck:
    assert zcheck.testzip() is None, 'ZIP CRC check failed!'

print(f'Packaged {len(added)} verified deliverables -> {zip_path} ({zip_path.stat().st_size:,} bytes)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
